# 03/05 — Permutation null for the rescue slope

Permute the BRI/PBS labels at the *mouse* level (WT held out — n=1 is fixed). Recompute the rescue slope per region. With 3 BRICHOS vs 7 PBS mice we have C(10,3) = 120 unique permutations, enough resolution for an empirical p down to ~1/120.

Output: `results/tables/attenuation/rescue_permutation_null.tsv`.

In [ ]:
from __future__ import annotations
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# regional-annotation h5ad lives on the processing volume
BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_ORIENTED = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
H5AD_BASE     = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD = H5AD_ORIENTED if H5AD_ORIENTED.exists() else H5AD_BASE
COUNT_LAYER = 'counts'   # raw integer counts live here, not in .X

TBL = ROOT / 'results' / 'tables' / 'attenuation'
TBL.mkdir(parents=True, exist_ok=True)
FIG = ROOT / 'results' / 'figures' / 'manuscript'
FIG.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY    = 'sample_id'           # change to 'library_id' if obs uses that
REGION_KEY    = 'anatomical_region'   # adjust to your obs column for regions
TREATMENT_KEY = 'treatment'
print('h5ad        :', H5AD)
print('count layer :', COUNT_LAYER)


In [ ]:
from itertools import combinations
from utils.attenuation import pseudobulk_lfc, rescue_slope

counts = pd.read_csv(TBL / 'pseudobulk_counts.tsv',
                     sep='\t', index_col=0)
meta   = pd.read_csv(TBL / 'pseudobulk_meta.tsv',
                     sep='\t', index_col=0)
disease_df = pd.read_csv(TBL / 'disease_signature.tsv',
                          sep='\t', index_col=0)
obs_stats = pd.read_csv(TBL / 'regional_rescue_stats.tsv',
                         sep='\t', index_col=0)


In [ ]:
# Build the permutation set: keep WT fixed, swap BRI/PBS labels
# at the mouse level. Each unique partition of non-WT mice into
# (BRI, PBS) groups of original sizes is one permutation.
treat_by_sample = (meta.groupby('sample')['treatment']
                   .first())
non_wt = treat_by_sample[treat_by_sample != 'WT'].index.tolist()
n_bri = (treat_by_sample == 'BRICHOS').sum()
perms = list(combinations(non_wt, n_bri))
print(f'{len(perms)} unique BRI-vs-PBS reassignments')


In [ ]:
padj_thresh, lfc_thresh = 0.1, 0.5
disease_up = disease_df.index[(disease_df.lfc >  lfc_thresh) &
                               (disease_df.padj < padj_thresh)]
sig_mask = disease_df.index.isin(disease_up)
sig_series = pd.Series(sig_mask, index=disease_df.index)

def slope_for(meta_local, region):
    try:
        res = pseudobulk_lfc(counts, meta_local,
                             group_a='BRICHOS', group_b='PBS',
                             region=region, min_n=2)
    except ValueError:
        return np.nan
    return rescue_slope(disease_df.lfc, res.lfc,
                        sig_mask=sig_series, n_boot=0)['slope']


In [ ]:
regions = obs_stats.index.tolist()
rows = []
for region in regions:
    obs_slope = obs_stats.loc[region, 'slope']
    null_slopes = []
    for bri_set in perms:
        local = meta.copy()
        new_treat = local['treatment'].copy()
        non_wt_mask = new_treat != 'WT'
        new_treat[non_wt_mask] = np.where(
            local.loc[non_wt_mask, 'sample'].isin(bri_set),
            'BRICHOS', 'PBS')
        local['treatment'] = new_treat
        s = slope_for(local, region)
        if np.isfinite(s):
            null_slopes.append(s)
    null_slopes = np.array(null_slopes)
    if len(null_slopes) == 0 or not np.isfinite(obs_slope):
        rows.append(dict(region=region, obs_slope=obs_slope,
                         null_median=np.nan, p_one_sided=np.nan,
                         n_perms=0))
        continue
    # rescue test: smaller (more negative) slope is rare under null
    p = (null_slopes <= obs_slope).mean()
    rows.append(dict(region=region, obs_slope=obs_slope,
                     null_median=float(np.nanmedian(null_slopes)),
                     p_one_sided=float(p),
                     n_perms=int(len(null_slopes))))
perm_df = pd.DataFrame(rows).set_index('region')
perm_df.to_csv(TBL / 'rescue_permutation_null.tsv', sep='\t')
perm_df
